In [ ]:
import numpy as np
import sympy as sp


# STAGE 1: Real-world data -> matrix representation

# Page 0 links to: 1, 2
# Page 1 links to: 2
# Page 2 links to: 0, 3, 4
# Page 3 links to: 4
# Page 4 links to: 0

num_pages = 5
adjacency_matrix = np.array([
    [0, 1, 1, 0, 0],
    [0, 0, 1, 0, 0],
    [1, 0, 0, 1, 1],
    [0, 0, 0, 0, 1],
    [1, 0, 0, 0, 0]
], dtype=float)

# Column-stochastic transition matrix: M[i, j] = chance of going from page j to page i
M = np.zeros((num_pages, num_pages))
for j in range(num_pages):
    num_outlinks = np.sum(adjacency_matrix[j])
    if num_outlinks > 0:
        M[:, j] = adjacency_matrix[j] / num_outlinks
    else:
        M[:, j] = 1.0 / num_pages  # dangling node

print("STAGE 1: Matrix representation")
print("Link matrix M:\n", np.round(M, 3))
print("Column sums (should all be 1):", M.sum(axis=0))


# STAGE 2: Matrix simplification (RREF of M - I)

A = sp.Matrix(M - np.eye(num_pages)).applyfunc(
    lambda x: sp.nsimplify(x, rational=True, tolerance=1e-10))
rref_matrix, pivot_cols = A.rref()

print("\nSTAGE 2: RREF of (M - I)")
sp.pprint(rref_matrix)
print("Pivot columns:", pivot_cols)


# STAGE 3: Structure of the space (rank, nullity, null space basis)

rank = np.linalg.matrix_rank(M - np.eye(num_pages))
nullity = num_pages - rank
null_basis = A.nullspace()[0]
null_basis = np.array(null_basis, dtype=float).flatten()
null_basis_normalized = null_basis / null_basis.sum()

print("\nSTAGE 3: Structure of the space")
print(f"Rank = {rank}, Nullity = {nullity}  (rank + nullity = {rank + nullity} = number of columns)")
print("Null space basis vector:", np.round(null_basis, 4))
print("Normalized (undamped PageRank):", np.round(null_basis_normalized, 4))


# Google matrix: G = d*M + (1-d)/N * J

d = 0.85
teleport = np.ones((num_pages, num_pages)) / num_pages
G = d * M + (1 - d) * teleport


# STAGE 4: Pattern discovery (eigenvalues & eigenvectors of G)

eigenvalues, eigenvectors = np.linalg.eig(G)
order = np.argsort(-np.abs(eigenvalues))  # largest magnitude first
eigenvalues = eigenvalues[order]
eigenvectors = eigenvectors[:, order]

print("\nSTAGE 4: Eigenvalues of G (sorted by magnitude)")
for k, lam in enumerate(eigenvalues):
    print(f"  lambda_{k+1} = {np.round(lam, 4)},  |lambda| = {abs(lam):.4f}")
print(f"Dominant eigenvalue = {eigenvalues[0].real:.4f}; second largest |lambda| = {abs(eigenvalues[1]):.4f} (<= d = {d})")

ranking_vector_eig = np.real(eigenvectors[:, 0])
ranking_vector_eig = ranking_vector_eig / np.sum(ranking_vector_eig)


# STAGE 5: System simplification (diagonalization G = P D P^-1)

P = eigenvectors
D = np.diag(eigenvalues)
G_rebuilt = P @ D @ np.linalg.inv(P)

print("\nSTAGE 5: Diagonalization")
print("G equals P D P^-1:", np.allclose(G, G_rebuilt))

# G^k = P D^k P^-1, so non-dominant parts shrink like |lambda|^k
v0 = np.ones(num_pages) / num_pages
for k in [1, 5, 10, 50]:
    Dk = np.diag(eigenvalues ** k)
    vk = np.real(P @ Dk @ np.linalg.inv(P) @ v0)
    err = np.linalg.norm(vk - ranking_vector_eig, 1)
    print(f"  k = {k:>2}: distance from PageRank = {err:.2e}")


# Power iteration (practical way to compute it)

v = np.ones(num_pages) / num_pages
tolerance = 1e-8
max_iterations = 200
converged = False

for i in range(max_iterations):
    v_next = G @ v
    if np.linalg.norm(v_next - v, 1) < tolerance:
        v = v_next
        print(f"\nPower iteration converged in {i+1} iterations.")
        converged = True
        break
    v = v_next
if not converged:
    print("\nWarning: power iteration did not converge.")


# FINAL OUTPUT

print("\nFinal PageRank Scores:")
for page_id, score in enumerate(v):
    print(f"  Page {page_id}: {score:.5f} (Eigenvector method: {ranking_vector_eig[page_id]:.5f})")

ranked_indices = np.argsort(v)[::-1]
print("\nPage Ranking order (Highest to Lowest):")
for rank_pos, page_id in enumerate(ranked_indices):
    print(f"  Rank {rank_pos+1}: Page {page_id} with score {v[page_id]:.5f}")

STAGE 1: Matrix representation
Link matrix M:
 [[0.    0.    0.333 0.    1.   ]
 [0.5   0.    0.    0.    0.   ]
 [0.5   1.    0.    0.    0.   ]
 [0.    0.    0.333 0.    0.   ]
 [0.    0.    0.333 1.    0.   ]]
Column sums (should all be 1): [1. 1. 1. 1. 1.]

STAGE 2: RREF of (M - I)
⎡1  0  0  0  -3/2⎤
⎢                ⎥
⎢0  1  0  0  -3/4⎥
⎢                ⎥
⎢0  0  1  0  -3/2⎥
⎢                ⎥
⎢0  0  0  1  -1/2⎥
⎢                ⎥
⎣0  0  0  0   0  ⎦
Pivot columns: (0, 1, 2, 3)

STAGE 3: Structure of the space
Rank = 4, Nullity = 1  (rank + nullity = 5 = number of columns)
Null space basis vector: [1.5  0.75 1.5  0.5  1.  ]
Normalized (undamped PageRank): [0.2857 0.1429 0.2857 0.0952 0.1905]

STAGE 4: Eigenvalues of G (sorted by magnitude)
  lambda_1 = (1+0j),  |lambda| = 1.0000
  lambda_2 = (-0+0.601j),  |lambda| = 0.6010
  lambda_3 = (-0-0.601j),  |lambda| = 0.6010
  lambda_4 = (-0.425+0.2454j),  |lambda| = 0.4907
  lambda_5 = (-0.425-0.2454j),  |lambda| = 0.4907
Dominant eigenval